In [13]:
import pandas as pd
import sys
import os
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()

sys.path.append(os.path.abspath("../.."))
from optimizer import variables

sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [14]:
historical_price_df = pd.read_csv('2_Processed_data/historical_price.csv')
historical_generation_df = pd.read_csv('2_Processed_data/historical_generation.csv')
future_price_df = pd.read_csv('2_Processed_data/future_price.csv')

In [15]:
def run_monte_carlo_sampling_type_1(df, variable):
    n_samples = 1000
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df[variable] = pd.to_numeric(df[variable], errors='coerce').round(2)
    years = sorted(df['Date'].dt.year.unique())
    target_hours = 8760
    
    np.random.seed(42)
    
    sample_years = np.zeros((target_hours, n_samples))
    
    for sim_num in tqdm(range(n_samples), desc="Simulations"):
        year_arr = []
        # for each 1 sample, iterate 12 times
        for month in range(1, 13):
            # get a random year
            random_year = np.random.choice(years)
            
            # Get the matching month and the random year
            month_arr = df[
                (df['Date'].dt.year == random_year) &
                (df['Date'].dt.month == month)
            ].sort_values('Date')[variable].to_numpy()
            ### This will be a month of values at a time
            
            # Add the month of values to 'year_arr'
            year_arr.extend(month_arr)
        
        
        year_arr = np.array(year_arr)
        
        if len(year_arr) > target_hours:
            year_arr = year_arr[:target_hours]
        # Store each sampled year as a column
        sample_years[:, sim_num] = year_arr
    
    # Clip negative generation, keep negative prices
    if variable == "Generation":
        sample_years = np.clip(sample_years, a_min=0, a_max=None)

    sample_years_df = pd.DataFrame(sample_years, columns=[f"Year {i + 1}" for i in range(n_samples)])
    sample_years_df.to_csv(f"2_Processed_data/resampled_{variable.lower()}.csv", index=False)


def _run_sampling_task(task):
    return run_monte_carlo_sampling_type_1(df=task["df"], variable=task["variable"])


tasks = [
    {"df": historical_price_df, "variable": "Price"},
    {"df": historical_generation_df, "variable": "Generation"},
]

# results = run_parallel(
#     function=_run_sampling_task,
#     data=tasks
# )


In [16]:
def run_monte_carlo_sampling_type_2():
    
    def load_sampling_superset(df=future_price_df):
        superset = df.copy()
        superset['date'] = pd.to_datetime(superset['Date'])
        superset["price"] = pd.to_numeric(superset["Price"], errors='coerce').round(2)
        return superset

    superset = load_sampling_superset()
    number_of_samples_to_perform = 1000
    np.random.seed(42)

    # Build the exact target timeline: one year before operation start through operation end.
    start_date = pd.to_datetime(variables.operation_start_date, dayfirst=True) - pd.DateOffset(years=1)
    end_date = pd.to_datetime(variables.operation_end_date, dayfirst=True)
    freq = f"{variables.operation_granularity_in_minutes}min"
    target_date_index = pd.date_range(start=start_date, end=end_date, freq=freq)
    target_periods = len(target_date_index)

    # Pre-build lookup dict once to avoid repeated DataFrame filtering in inner loop.
    available_years = superset.date.dt.year.unique()
    price_lookup = {
        (int(year), int(month), int(hour)): group['price'].values
        for (year, month, hour), group in superset.groupby(
            [superset.date.dt.year, superset.date.dt.month, superset.date.dt.hour]
        )
    }

    samples = []
    for sample_number in tqdm(range(number_of_samples_to_perform)):
        sample_data = []

        # Keep generating months until we have exactly the required number of points.
        while len(sample_data) < target_periods:
            for right_cycle_month in range(1, 13):
                random_year = int(np.random.choice(available_years))
                days_in_month = pd.Timestamp(year=random_year, month=right_cycle_month, day=1).days_in_month

                # Sample all days for each hour at once, then interleave into chronological order.
                month_data = np.array([
                    np.random.choice(price_lookup[(random_year, right_cycle_month, h)], size=days_in_month)
                    for h in range(24)
                ]).T.flatten()  # shape (days_in_month, 24) -> row-major = chronological

                remaining = target_periods - len(sample_data)
                sample_data.extend(month_data[:remaining])

                if len(sample_data) >= target_periods:
                    break

        samples.append(sample_data)

    samples = pd.DataFrame(np.array(samples).T, columns=[f"Sample {i + 1}" for i in range(number_of_samples_to_perform)])
    samples['Date'] = target_date_index
    samples.to_csv("2_Processed_data/resampled_aurora_price.csv", index=False)
    return samples

# samples = run_monte_carlo_sampling_type_2()
# display(samples[:10])


In [17]:
# display(samples[-10:])

In [18]:
def create_bess_designs(
    duration_low_h,
    duration_high_h,
    duration_step_h,
    power_low_mw,
    power_high_mw,
    power_step_mw,
    interval_minutes,
    usable_fraction,
    output_csv_path="2_Processed_data/bess_designs.csv",
    round_dp=4,
):

    def _inclusive_range(low, high, step, ndigits):
        values = []
        current = float(low)
        eps = 10 ** (-(ndigits + 2))
        while current <= float(high) + eps:
            values.append(round(current, ndigits))
            current += float(step)
        return np.array(values, dtype=float)

    durations = _inclusive_range(duration_low_h, duration_high_h, duration_step_h, round_dp)
    powers = _inclusive_range(power_low_mw, power_high_mw, power_step_mw, round_dp)

    rows = []
    interval_hours = interval_minutes / 60.0

    for duration_h in durations:
        for power_mw in powers:
            bess_mwh = power_mw * duration_h * usable_fraction
            bess_mwh_per_interval = power_mw * interval_hours

            rows.append({
                "BESS_Duration_h": round(float(duration_h), round_dp),
                "BESS_Power_MW": round(float(power_mw), round_dp),
                "BESS_MWh": round(float(bess_mwh), round_dp),
                "BESS_MWh_per_interval": round(float(bess_mwh_per_interval), round_dp),
            })

    bess_designs_df = pd.DataFrame(rows).drop_duplicates().sort_values(
        ["BESS_Duration_h", "BESS_Power_MW"]
    ).reset_index(drop=True)

    bess_designs_df.to_csv(output_csv_path, index=False)

    return bess_designs_df


bess_designs_df = create_bess_designs(
    duration_low_h=2,
    duration_high_h=8.0,
    duration_step_h=0.5,
    power_low_mw=4.96,
    power_high_mw=4.96,
    power_step_mw=4.96,
    interval_minutes=variables.operation_granularity_in_minutes,
    usable_fraction=variables.bess_hours_usable_fraction,
    output_csv_path="2_Processed_data/bess_designs.csv",
)

bess_designs_df

,BESS_Duration_h,BESS_Power_MW,BESS_MWh,BESS_MWh_per_interval
0,2.0,4.96,9.3820,4.96
1,2.5,4.96,11.7275,4.96
2,3.0,4.96,14.0730,4.96
3,3.5,4.96,16.4185,4.96
4,4.0,4.96,18.7639,4.96
5,4.5,4.96,21.1094,4.96
6,5.0,4.96,23.4549,4.96
7,5.5,4.96,25.8004,4.96
8,6.0,4.96,28.1459,4.96
9,6.5,4.96,30.4914,4.96


In [19]:
def create_scaled_generation_sets(
    power_low_mw,        
    power_high_mw,         
    power_step_mw,        
    input_csv_path,        
    output_csv_path,      
    round_dp=4,           
):

    # Create an inclusive float range helper (includes high bound when aligned).
    def _inclusive_range(low, high, step, ndigits):
        values = []                        # Collect generated values.
        current = float(low)               # Start at lower bound.
        eps = 10 ** (-(ndigits + 2))       # Tiny tolerance for float comparisons.
        while current <= float(high) + eps:
            values.append(round(current, ndigits))  # Store rounded value.
            current += float(step)         # Move to next value by step.
        return np.array(values, dtype=float)  # Return as numpy array for iteration.

    generation_df = pd.read_csv(input_csv_path)
    base_generation = pd.to_numeric(generation_df["Generation"], errors="coerce")
    base_generation = np.clip(base_generation.to_numpy(dtype=float), a_min=0.0, a_max=None)
    max_generation = float(np.max(base_generation)) if len(base_generation) else 0.0
    normalized_profile = np.clip(base_generation / max_generation, a_min=0.0, a_max=1.0)
    power_values = _inclusive_range(power_low_mw, power_high_mw, power_step_mw, round_dp)

    scaled = {}
    for power_mw in power_values:
        col_name = f"{power_mw:g}"             
        scaled[col_name] = np.round(normalized_profile * float(power_mw), round_dp)  

    scaled_df = pd.DataFrame(scaled)
    scaled_df.to_csv(output_csv_path, index=False)
    return scaled_df


# scaled_generation_df = create_scaled_generation_sets(
#     power_low_mw=0.0,
#     power_high_mw=7.0,
#     power_step_mw=1.0,
#     input_csv_path="2_Processed_data/generation.csv",
#     output_csv_path="2_Processed_data/resampled_generation_scaled.csv",
# )

# scaled_generation_df[:100]